# TechStore Perú S.A.C. — Generador de datos sintéticos (Data Warehouse)

Este notebook genera de forma **reproducible** (semilla `42`) las 6 tablas del modelo dimensional de TechStore Perú S.A.C.: `Dim_Cliente`, `Dim_Producto`, `Dim_Tienda`, `Dim_Tiempo`, `Dim_Promocion` y `Fact_Ventas`.

**Flujo del notebook:**
1. Configuración y semillas.
2. Generación de las 5 tablas de dimensión.
3. Generación de `Fact_Ventas` con estacionalidad, concentración tipo Pareto, afinidad de canasta y churn.
4. Inyección de problemas de calidad (solo en las versiones que se guardan en `data/raw/`).
5. Exportación a CSV (`;` como separador, codificación UTF-8).
6. Resumen de calidad inicial (filas, % nulos, duplicados, claves huérfanas).


In [10]:
!pip install Faker

In [12]:
import numpy as np
import pandas as pd
from faker import Faker
import random
import os

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker('es_ES')
Faker.seed(SEED)

RAW_DIR = "data/raw"
PROC_DIR = "data/processed"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)


## 1. Configuración inicial
Importamos librerías, fijamos las semillas de `numpy` y `Faker` en `42` para que la generación sea 100% reproducible, configuramos `Faker` con locale `es_ES`, y preparamos las carpetas de salida `data/raw/` y `data/processed/`.

## 2. Dim_Tiempo (730 días — 2 años)
Generamos el calendario completo desde 2024-01-01, con atributos de fecha (día, mes, trimestre, año, día de la semana en español) y una bandera `es_feriado` basada en feriados peruanos.

También calculamos un **peso estacional interno** (`_peso_estacional`, columna auxiliar que NO se exporta) más alto en Cyber Days (marzo/abril y octubre), Black Friday (noviembre) y Navidad (diciembre), y más bajo en meses de baja demanda (febrero, junio). Este peso se usará más adelante para muestrear las fechas de `Fact_Ventas` con estacionalidad realista.

In [13]:
# ---------------------------------------------------------------
# DIM_TIEMPO
# ---------------------------------------------------------------
FECHA_INICIO = pd.Timestamp("2024-01-01")
N_DIAS = 730
fechas = pd.date_range(FECHA_INICIO, periods=N_DIAS, freq="D")

feriados_fijos = {
    (1, 1), (5, 1), (6, 7), (7, 28), (7, 29), (8, 30),
    (10, 8), (11, 1), (12, 8), (12, 25),
}

def es_feriado(f):
    return (f.month, f.day) in feriados_fijos

dim_tiempo = pd.DataFrame({
    "fecha": fechas,
    "dia": fechas.day,
    "mes": fechas.month,
    "trimestre": fechas.quarter,
    "anio": fechas.year,
    "dia_semana": fechas.day_name(locale="es_ES.utf8") if False else fechas.dayofweek,
    "es_feriado": [1 if es_feriado(f) else 0 for f in fechas],
})

dias_semana_es = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves",
                   4: "Viernes", 5: "Sábado", 6: "Domingo"}
dim_tiempo["dia_semana"] = fechas.dayofweek.map(dias_semana_es)

# campaign flag used later for seasonality weighting (not exported as a column,
# only used internally to bias Fact_Ventas sampling)
def campaign_weight(f):
    w = 1.0
    if f.month in (3, 4) and 15 <= f.day <= 30:
        w *= 2.2  # Cyber Wow / Cyber Days
        # note: kept simple, real Cyber Days vary by year
    if f.month == 10 and 15 <= f.day <= 31:
        w *= 2.0  # Cyber Days Octubre
    if f.month == 11 and 20 <= f.day <= 30:
        w *= 3.0  # Black Friday
    if f.month == 12:
        w *= 2.5  # Campaña Navideña
        if f.day >= 20:
            w *= 1.5
    if f.month in (2, 6):
        w *= 0.6  # meses de baja demanda
    return w

dim_tiempo["_peso_estacional"] = dim_tiempo["fecha"].apply(campaign_weight)

print("Dim_Tiempo:", dim_tiempo.shape)


Dim_Tiempo: (730, 8)


## 3. Dim_Cliente (5,000 clientes)
Generamos nombres con `Faker` (coherentes con el sexo asignado), fecha de nacimiento (edad 18-70), distrito (distritos de Lima/Callao), fecha de alta al programa, y segmento de fidelización (Bronce/Plata/Oro/Platino con distribución realista).

Además calculamos `frecuencia_pesos`: un peso de compra por cliente con distribución Pareto, para que pocos clientes sean muy frecuentes y la mayoría compre esporádicamente. Este peso se usa más adelante al muestrear `Fact_Ventas`.

In [14]:
# ---------------------------------------------------------------
# DIM_CLIENTE
# ---------------------------------------------------------------
N_CLIENTES = 5000

distritos_lima = [
    "San Isidro", "Miraflores", "Surco", "La Molina", "San Borja",
    "Jesús María", "Lince", "Magdalena", "Pueblo Libre", "San Miguel",
    "Los Olivos", "San Juan de Lurigancho", "Comas", "Ate", "Chorrillos",
    "Barranco", "Villa El Salvador", "Villa María del Triunfo", "Callao",
    "Santiago de Surco",
]

segmentos = ["Bronce", "Plata", "Oro", "Platino"]
segmento_pesos = [0.45, 0.30, 0.18, 0.07]

sexo_arr = np.random.choice(["M", "F"], size=N_CLIENTES, p=[0.49, 0.51])
nombres = [fake.name_male() if s == "M" else fake.name_female() for s in sexo_arr]

fecha_nacimiento = [
    fake.date_of_birth(minimum_age=18, maximum_age=70) for _ in range(N_CLIENTES)
]

fecha_alta = [
    fake.date_between(start_date=pd.Timestamp("2021-01-01").date(),
                       end_date=pd.Timestamp("2025-12-25").date())
    for _ in range(N_CLIENTES)
]

dim_cliente = pd.DataFrame({
    "id_cliente": np.arange(1, N_CLIENTES + 1),
    "nombre": nombres,
    "sexo": sexo_arr,
    "fecha_nacimiento": fecha_nacimiento,
    "distrito": np.random.choice(distritos_lima, size=N_CLIENTES),
    "fecha_alta": fecha_alta,
    "segmento_programa": np.random.choice(segmentos, size=N_CLIENTES, p=segmento_pesos),
})

# peso de frecuencia de compra por cliente (Pareto/lognormal): pocos clientes
# muy frecuentes y muchos esporadicos. Usado luego para muestrear Fact_Ventas.
frecuencia_pesos = np.random.pareto(a=1.6, size=N_CLIENTES) + 0.05
frecuencia_pesos = frecuencia_pesos / frecuencia_pesos.sum()

print("Dim_Cliente:", dim_cliente.shape)


Dim_Cliente: (5000, 7)


## 4. Dim_Producto (500 productos)
Definimos un catálogo propio de tienda de tecnología con 14 categorías (Laptops, Computadoras, Celulares, Tablets, Monitores, Componentes, Perifericos, Audio, Gaming, Redes, Almacenamiento, Impresion, Accesorios, Hogar Inteligente), cada una con subcategorías y marcas reconocidas del rubro. Para cada producto se genera un nombre, precio de lista y costo (con un margen aleatorio realista).

También calculamos `_peso_popularidad` (columna auxiliar interna) con distribución Pareto, de modo que ~20% de los productos concentre ~80% de las ventas al muestrear `Fact_Ventas`.

In [15]:
# ---------------------------------------------------------------
# DIM_PRODUCTO
# ---------------------------------------------------------------
N_PRODUCTOS = 500

catalogo = {
    "Laptops": {"sub": ["Ultrabook", "Gaming", "Convertible", "Empresarial"],
                "marcas": ["HP", "Dell", "Lenovo", "Asus", "Acer", "Apple"],
                "precio": (1500, 8000)},
    "Computadoras": {"sub": ["Desktop", "All in One", "Mini PC"],
                      "marcas": ["HP", "Dell", "Lenovo", "Asus"],
                      "precio": (1200, 6000)},
    "Celulares": {"sub": ["Gama Alta", "Gama Media", "Gama Baja"],
                  "marcas": ["Samsung", "Apple", "Xiaomi", "Motorola", "Huawei"],
                  "precio": (400, 6000)},
    "Tablets": {"sub": ["Android", "iPadOS"],
                "marcas": ["Samsung", "Apple", "Lenovo", "Huawei"],
                "precio": (600, 4000)},
    "Monitores": {"sub": ["Gaming", "Profesional", "Curvo", "Estandar"],
                  "marcas": ["LG", "Samsung", "AOC", "Dell", "ASUS"],
                  "precio": (500, 3500)},
    "Componentes": {"sub": ["Procesadores", "Tarjetas de Video", "Placas Madre",
                             "Memorias RAM", "Fuentes de Poder"],
                     "marcas": ["Intel", "AMD", "NVIDIA", "Corsair", "Kingston"],
                     "precio": (200, 5000)},
    "Perifericos": {"sub": ["Teclados", "Mouse", "Webcams", "Combos"],
                     "marcas": ["Logitech", "Razer", "HyperX", "Redragon"],
                     "precio": (40, 800)},
    "Audio": {"sub": ["Audifonos", "Parlantes", "Barras de Sonido"],
              "marcas": ["JBL", "Sony", "Bose", "Logitech"],
              "precio": (50, 1500)},
    "Gaming": {"sub": ["Consolas", "Sillas Gamer", "Volantes", "Controles"],
               "marcas": ["Sony", "Microsoft", "Nintendo", "Razer"],
               "precio": (150, 3000)},
    "Redes": {"sub": ["Routers", "Repetidores", "Switches", "Tarjetas de Red"],
              "marcas": ["TP-Link", "Netgear", "Ubiquiti", "D-Link"],
              "precio": (60, 900)},
    "Almacenamiento": {"sub": ["Discos SSD", "Discos HDD", "Pendrives", "Memorias microSD"],
                        "marcas": ["Kingston", "Western Digital", "Seagate", "SanDisk"],
                        "precio": (40, 1200)},
    "Impresion": {"sub": ["Impresoras Laser", "Impresoras Tinta", "Multifuncionales", "Toners"],
                  "marcas": ["HP", "Epson", "Canon", "Brother"],
                  "precio": (200, 2500)},
    "Accesorios": {"sub": ["Fundas", "Protectores de Pantalla", "Cables", "Mochilas", "Soportes"],
                   "marcas": ["Belkin", "Anker", "Targus", "Xtech"],
                   "precio": (15, 400)},
    "Hogar Inteligente": {"sub": ["Asistentes de Voz", "Camaras de Seguridad",
                                   "Focos Inteligentes", "Enchufes Inteligentes"],
                           "marcas": ["Amazon", "Google", "TP-Link", "Xiaomi"],
                           "precio": (60, 1200)},
}

categorias_list = list(catalogo.keys())
# pesos de categoria: mas productos en categorias de alta rotacion
categoria_peso_prod = np.array([14, 6, 16, 6, 8, 8, 10, 7, 6, 5, 6, 4, 12, 4], dtype=float)
categoria_peso_prod = categoria_peso_prod / categoria_peso_prod.sum()

productos_rows = []
for i in range(1, N_PRODUCTOS + 1):
    cat = np.random.choice(categorias_list, p=categoria_peso_prod)
    info = catalogo[cat]
    sub = random.choice(info["sub"])
    marca = random.choice(info["marcas"])
    precio_min, precio_max = info["precio"]
    precio_lista = round(np.random.uniform(precio_min, precio_max), 2)
    margen = np.random.uniform(0.55, 0.80)  # costo como fraccion del precio
    costo = round(precio_lista * margen, 2)
    modelo = fake.bothify(text="??-####").upper()
    nombre_prod = f"{marca} {sub} {modelo}"
    productos_rows.append((i, nombre_prod, cat, sub, marca, precio_lista, costo))

dim_producto = pd.DataFrame(
    productos_rows,
    columns=["id_producto", "nombre", "categoria", "subcategoria", "marca",
             "precio_lista", "costo"],
)

# popularidad Pareto de productos: ~20% de productos concentra ~80% de las ventas
pop_raw = np.random.pareto(a=1.2, size=N_PRODUCTOS) + 0.1
producto_pesos = pop_raw / pop_raw.sum()
dim_producto["_peso_popularidad"] = producto_pesos

print("Dim_Producto:", dim_producto.shape)


Dim_Producto: (500, 8)


## 5. Dim_Tienda (15 tiendas)
Red de tiendas físicas en varias regiones del Perú (Lima, Callao, Arequipa, La Libertad, Lambayeque, Cusco) y 3 canales online.

In [16]:
# ---------------------------------------------------------------
# DIM_TIENDA
# ---------------------------------------------------------------
tiendas_info = [
    (1, "TechStore San Isidro", "fisico", "Lima", "Lima"),
    (2, "TechStore Miraflores", "fisico", "Lima", "Lima"),
    (3, "TechStore Jockey Plaza", "fisico", "Lima", "Lima"),
    (4, "TechStore Mega Plaza", "fisico", "Lima", "Lima"),
    (5, "TechStore Plaza Norte", "fisico", "Lima", "Lima"),
    (6, "TechStore Salaverry", "fisico", "Lima", "Lima"),
    (7, "TechStore La Molina", "fisico", "Lima", "Lima"),
    (8, "TechStore Callao", "fisico", "Callao", "Callao"),
    (9, "TechStore Arequipa Real Plaza", "fisico", "Arequipa", "Arequipa"),
    (10, "TechStore Trujillo Mall Aventura", "fisico", "La Libertad", "Trujillo"),
    (11, "TechStore Chiclayo Real Plaza", "fisico", "Lambayeque", "Chiclayo"),
    (12, "TechStore Cusco", "fisico", "Cusco", "Cusco"),
    (13, "TechStore Online Perú", "online", "Lima", "Lima"),
    (14, "TechStore App Marketplace", "online", "Lima", "Lima"),
    (15, "TechStore Outlet Online", "online", "Lima", "Lima"),
]
dim_tienda = pd.DataFrame(
    tiendas_info, columns=["id_tienda", "nombre", "canal", "region", "ciudad"]
)
print("Dim_Tienda:", dim_tienda.shape)


Dim_Tienda: (15, 5)


## 6. Dim_Promocion (40 promociones)
Campañas típicas de una tienda de tecnología (Cyber Days, Black Friday, Cyber Monday, Navidad, Aniversario, Fiestas Patrias, Regreso a Clases, etc.). Las ventanas de vigencia de las primeras 12 promociones se alinean intencionalmente con los picos de estacionalidad definidos en `Dim_Tiempo`, para que las ventas y los descuentos sean consistentes entre sí.

In [17]:
# ---------------------------------------------------------------
# DIM_PROMOCION
# ---------------------------------------------------------------
N_PROMOCIONES = 40
promo_tipos = ["Descuento %", "2x1", "Cupón", "Bundle", "Liquidación"]
nombres_campania = [
    "Cyber Days Marzo", "Cyber Wow", "Cyber Days Octubre", "Black Friday",
    "Cyber Monday", "Navidad TechStore", "Aniversario TechStore",
    "Fiestas Patrias", "Regreso a Clases", "Liquidación de Temporada",
    "Semana del Gamer", "Descuento Flash",
]

# ventanas de campania alineadas con la estacionalidad de Fact_Ventas
ventanas_campania = [
    ("2024-03-18", "2024-03-24"), ("2024-04-01", "2024-04-07"),
    ("2024-10-15", "2024-10-22"), ("2024-11-22", "2024-11-29"),
    ("2024-12-01", "2024-12-24"), ("2024-07-20", "2024-07-29"),
    ("2025-03-17", "2025-03-23"), ("2025-04-01", "2025-04-07"),
    ("2025-10-14", "2025-10-21"), ("2025-11-21", "2025-11-28"),
    ("2025-12-01", "2025-12-24"), ("2025-07-20", "2025-07-29"),
]

promo_rows = []
for i in range(1, N_PROMOCIONES + 1):
    nombre_p = f"{random.choice(nombres_campania)} {random.choice(['Tech','Gamer','Hogar','Total'])} {i}"
    tipo = random.choice(promo_tipos)
    descuento_pct = int(np.random.choice([10, 15, 20, 25, 30, 35, 40], p=[.2,.2,.2,.15,.1,.1,.05]))
    if i <= len(ventanas_campania):
        ini, fin = ventanas_campania[i - 1]
        f_ini = pd.Timestamp(ini)
        f_fin = pd.Timestamp(fin)
    else:
        f_ini = FECHA_INICIO + pd.Timedelta(days=int(np.random.uniform(0, N_DIAS - 10)))
        f_fin = f_ini + pd.Timedelta(days=int(np.random.uniform(3, 10)))
    promo_rows.append((i, nombre_p, tipo, descuento_pct, f_ini, f_fin))

dim_promocion = pd.DataFrame(
    promo_rows,
    columns=["id_promocion", "nombre", "tipo", "descuento_pct", "fecha_inicio", "fecha_fin"],
)
print("Dim_Promocion:", dim_promocion.shape)


Dim_Promocion: (40, 6)


## 7. Fact_Ventas (60,000 líneas de venta)
Esta es la celda más importante del notebook. La generación de cada línea de venta sigue esta lógica:

- **Estacionalidad**: la fecha de cada venta se muestrea usando el peso estacional de `Dim_Tiempo`, por lo que hay mucho más volumen en Cyber Days, Black Friday y Navidad.
- **Churn / abandono**: ~22.5% de los clientes se marcan como "churn"; estos clientes solo pueden generar ventas antes de una fecha de corte (65% del periodo), simulando que dejaron de comprar.
- **Concentración Pareto**: el producto principal de cada línea se muestrea con el peso de popularidad de `Dim_Producto` (20% de productos → ~80% de las ventas), y el cliente se muestrea con su peso de frecuencia (pocos clientes muy frecuentes).
- **Afinidad de canasta**: cada "orden" (mismo cliente, fecha y tienda) puede tener de 1 a 4 líneas. Si el producto principal es una laptop, celular, monitor o teclado, se privilegia agregar un producto complementario (mouse, mochila, funda, protector de pantalla) para generar reglas de asociación con `lift > 1` (laptop+mouse, teclado+mouse, celular+funda, celular+protector, laptop+mochila, monitor+teclado).
- **Promoción vigente**: se busca si existe una promoción activa en la fecha de la venta; si la hay, se aplica su `id_promocion` y su `descuento_pct` a las líneas de esa orden; si no, el descuento es 0.
- **precio_unitario**: varía ±8% alrededor del `precio_lista` del producto.

Al final, agregamos la columna analítica `abandono` (0/1) a una copia limpia de `Dim_Cliente` (`dim_cliente_clean`), que sirve como etiqueta para un futuro modelo de clasificación de churn.

In [19]:
# ---------------------------------------------------------------
# FACT_VENTAS
# ---------------------------------------------------------------
N_LINEAS_OBJETIVO = 60000

# --- 1) elegir clientes que compran (churn) ---------------------
# ~20-25% de clientes "abandonan": su ultima compra ocurre antes de
# una fecha de corte (dejan de comprar en el segundo semestre final).
pct_churn = 0.225
es_churn_cliente = np.random.rand(N_CLIENTES) < pct_churn
fecha_corte_churn = FECHA_INICIO + pd.Timedelta(days=int(N_DIAS * 0.65))

# --- 2) pesos de fecha (estacionalidad) --------------------------
pesos_fecha = dim_tiempo["_peso_estacional"].values
pesos_fecha = pesos_fecha / pesos_fecha.sum()
fecha_idx_array = dim_tiempo.index.values

# --- 3) armar "ordenes" (canastas) hasta acumular >= N lineas ---
# cada orden = mismo cliente, misma fecha, misma tienda, misma promocion
# con 1 a 4 lineas de producto, incorporando afinidad de canasta.

afinidad_categoria = {
    ("Laptops", "Perifericos"): 0.35,
    ("Perifericos_Teclados", "Perifericos_Mouse"): 0.4,
    ("Celulares", "Accesorios_Fundas"): 0.4,
    ("Celulares", "Accesorios_Protectores de Pantalla"): 0.35,
    ("Laptops", "Accesorios_Mochilas"): 0.3,
    ("Monitores", "Perifericos_Teclados"): 0.3,
}

productos_por_cat = dim_producto.groupby("categoria")
productos_por_subcat = dim_producto.groupby(["categoria", "subcategoria"])

def elegir_producto_afin(cat_principal, subcat_principal):
    """Devuelve un id_producto complementario segun reglas de afinidad, o None.

    Nota: cuando la canasta ya decidio tener >1 items (n_items > 1) y el
    producto principal pertenece a una categoria/subcategoria con afinidad
    conocida, se privilegia fuertemente el par complementario (laptop+mouse,
    teclado+mouse, celular+funda, celular+protector, laptop+mochila,
    monitor+teclado) para que la canasta resultante muestre reglas de
    asociacion con lift > 1.
    """
    candidatos = []
    if cat_principal == "Laptops":
        candidatos = dim_producto[dim_producto["subcategoria"].isin(["Mouse", "Mochilas"])]
    elif subcat_principal == "Teclados":
        candidatos = dim_producto[dim_producto["subcategoria"] == "Mouse"]
    elif cat_principal == "Celulares":
        candidatos = dim_producto[dim_producto["subcategoria"].isin(["Fundas", "Protectores de Pantalla"])]
    elif cat_principal == "Monitores":
        candidatos = dim_producto[dim_producto["subcategoria"] == "Teclados"]
    if len(candidatos) == 0:
        return None
    pesos = candidatos["_peso_popularidad"].values
    pesos = pesos / pesos.sum()
    return int(np.random.choice(candidatos["id_producto"].values, p=pesos))

id_cliente_arr = dim_cliente["id_cliente"].values
producto_ids = dim_producto["id_producto"].values
producto_pesos_norm = dim_producto["_peso_popularidad"].values
producto_pesos_norm = producto_pesos_norm / producto_pesos_norm.sum()
producto_by_id = dim_producto.set_index("id_producto")

tienda_ids = dim_tienda["id_tienda"].values
# canal online gana peso con el tiempo, pero simplificamos con pesos fijos
tienda_pesos = np.where(dim_tienda["canal"].values == "online", 1.4, 1.0)
tienda_pesos = tienda_pesos / tienda_pesos.sum()

promo_ini = dim_promocion["fecha_inicio"].values
promo_fin = dim_promocion["fecha_fin"].values
promo_ids = dim_promocion["id_promocion"].values
promo_pct = dim_promocion["descuento_pct"].values

def promocion_vigente(fecha):
    activas = np.where((promo_ini <= np.datetime64(fecha)) & (np.datetime64(fecha) <= promo_fin))[0]
    if len(activas) == 0:
        return None, 0
    sel = np.random.choice(activas)
    return int(promo_ids[sel]), int(promo_pct[sel])

filas_venta = []
id_venta = 1
lineas_generadas = 0

# clientes activos ponderados por frecuencia (Pareto de clientes)
clientes_pesos = frecuencia_pesos.copy()

while lineas_generadas < N_LINEAS_OBJETIVO:
    # elegir fecha con estacionalidad
    idx_fecha = np.random.choice(fecha_idx_array, p=pesos_fecha)
    fecha_venta = dim_tiempo.loc[idx_fecha, "fecha"]

    # elegir cliente; si es cliente "churn", solo puede comprar antes del corte
    intentos = 0
    while True:
        cli_idx = np.random.choice(N_CLIENTES, p=clientes_pesos)
        if not es_churn_cliente[cli_idx] or fecha_venta <= fecha_corte_churn:
            break
        intentos += 1
        if intentos > 5:
            break
    id_cli = int(id_cliente_arr[cli_idx])

    id_tda = int(np.random.choice(tienda_ids, p=tienda_pesos))
    id_promo, pct_promo = promocion_vigente(fecha_venta)

    # tamano de la canasta: 1 a 4 productos, sesgado a canastas pequenas
    n_items = np.random.choice([1, 2, 3, 4], p=[0.55, 0.28, 0.12, 0.05])

    productos_canasta = []
    id_prod_principal = int(np.random.choice(producto_ids, p=producto_pesos_norm))
    productos_canasta.append(id_prod_principal)
    prod_info = producto_by_id.loc[id_prod_principal]

    if n_items > 1:
        afin = elegir_producto_afin(prod_info["categoria"], prod_info["subcategoria"])
        if afin is not None:
            productos_canasta.append(afin)

    while len(productos_canasta) < n_items:
        extra = int(np.random.choice(producto_ids, p=producto_pesos_norm))
        productos_canasta.append(extra)

    for id_prod in productos_canasta:
        info_p = producto_by_id.loc[id_prod]
        precio_lista = info_p["precio_lista"]
        variacion = np.random.uniform(-0.08, 0.08)
        precio_unitario = round(precio_lista * (1 + variacion), 2)
        cantidad = int(np.random.choice([1, 2, 3], p=[0.75, 0.18, 0.07]))
        descuento_val = pct_promo if id_promo is not None else 0
        filas_venta.append((
            id_venta, fecha_venta, id_cli, id_prod, id_tda,
            id_promo, cantidad, precio_unitario, descuento_val,
        ))
        id_venta += 1
        lineas_generadas += 1
        if lineas_generadas >= N_LINEAS_OBJETIVO:
            break

fact_ventas = pd.DataFrame(
    filas_venta,
    columns=["id_venta", "fecha", "id_cliente", "id_producto", "id_tienda",
             "id_promocion", "cantidad", "precio_unitario", "descuento"],
)

print("Fact_Ventas (limpio, pre-defectos):", fact_ventas.shape)
print(fact_ventas.head())

# columna de abandono (churn) a nivel cliente, para clasificacion posterior
dim_cliente_clean = dim_cliente.copy()
dim_cliente_clean["abandono"] = es_churn_cliente.astype(int)
print(dim_cliente_clean["abandono"].value_counts(normalize=True))


Fact_Ventas (limpio, pre-defectos): (60000, 9)
   id_venta      fecha  id_cliente  id_producto  id_tienda  id_promocion  \
0         1 2024-02-02        4215          201         14           NaN   
1         2 2024-02-02        4215          426         14           NaN   
2         3 2024-02-02        4215           66         14           NaN   
3         4 2024-02-02        4215          208         14           NaN   
4         5 2025-12-25         472          208          6           NaN   

   cantidad  precio_unitario  descuento  
0         3          1960.52          0  
1         3           352.87          0  
2         1           630.58          0  
3         1          1728.29          0  
4         1          1697.97          0  
abandono
0    0.7768
1    0.2232
Name: proportion, dtype: float64


## 8. Inyección de problemas de calidad (solo en `data/raw/`)
A partir de aquí trabajamos siempre sobre **copias** de las tablas limpias, para preservar las versiones originales que se guardarán en `data/processed/`. Se introducen los siguientes defectos, tal como los tendría un extracto crudo real:

**`formato_fecha_mixto`**: función auxiliar que convierte una fecha a texto en uno de tres formatos inconsistentes: `dd/mm/aaaa`, `aaaa-mm-dd`, o texto en español ("20 de diciembre de 2024").

In [20]:
# =================================================================
# INYECCION DE PROBLEMAS DE CALIDAD (solo en data/raw)
# =================================================================

def formato_fecha_mixto(fecha, rng):
    """Devuelve la fecha como texto en uno de varios formatos inconsistentes."""
    f = pd.Timestamp(fecha)
    r = rng.random()
    if r < 0.40:
        return f.strftime("%d/%m/%Y")
    elif r < 0.80:
        return f.strftime("%Y-%m-%d")
    else:
        meses_es = ["enero", "febrero", "marzo", "abril", "mayo", "junio",
                    "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
        return f"{f.day} de {meses_es[f.month - 1]} de {f.year}"

rng_dates = np.random.default_rng(SEED)


### 8.1 Fact_Ventas — defectos
- **Claves huérfanas** (~0.5%): se reemplaza `id_producto` por IDs que no existen en `Dim_Producto` (rango 9001-9098).
- **Duplicados** (~1.5%): se duplican líneas de venta completas (mismo `id_venta` repetido).
- **Nulos en `descuento`** (~5%).
- **Outliers**: `cantidad` extremas (80-500 unidades) y `precio_unitario` multiplicado por un factor de 15x-40x en un subconjunto pequeño de filas.
- Se reordena aleatoriamente la tabla y, al final, se convierte `fecha` a texto en formatos mixtos.

In [21]:
# ---------- Fact_Ventas RAW ----------
fact_raw = fact_ventas.copy()

# 1) claves huerfanas: ~0.5% de id_producto que no existen en Dim_Producto
n_huerfanas = int(len(fact_raw) * 0.005)
idx_huerfanas = np.random.choice(fact_raw.index, size=n_huerfanas, replace=False)
ids_invalidos = np.random.randint(9001, 9099, size=n_huerfanas)
fact_raw.loc[idx_huerfanas, "id_producto"] = ids_invalidos

# 2) duplicar ~1.5% de las lineas de venta (filas completas repetidas)
n_dup = int(len(fact_raw) * 0.015)
filas_dup = fact_raw.sample(n=n_dup, random_state=SEED)
fact_raw = pd.concat([fact_raw, filas_dup], ignore_index=True)

# 3) valores faltantes ~5% en descuento
n_null_desc = int(len(fact_raw) * 0.05)
idx_null_desc = np.random.choice(fact_raw.index, size=n_null_desc, replace=False)
fact_raw.loc[idx_null_desc, "descuento"] = np.nan

# 4) outliers poco realistas en cantidad y precio_unitario (~0.3% cada uno)
n_out = max(1, int(len(fact_raw) * 0.003))
idx_out_cant = np.random.choice(fact_raw.index, size=n_out, replace=False)
fact_raw.loc[idx_out_cant, "cantidad"] = np.random.randint(80, 500, size=n_out)

idx_out_precio = np.random.choice(fact_raw.index, size=n_out, replace=False)
fact_raw.loc[idx_out_precio, "precio_unitario"] = fact_raw.loc[idx_out_precio, "precio_unitario"] * np.random.uniform(15, 40, size=n_out)

# 5) reordenar aleatoriamente (para que los duplicados no queden todos al final)
fact_raw = fact_raw.sample(frac=1, random_state=SEED).reset_index(drop=True)

# 6) fechas en formatos mixtos (texto)
fact_raw["fecha"] = fact_raw["fecha"].apply(lambda f: formato_fecha_mixto(f, rng_dates))

print("\nFact_Ventas RAW (con defectos):", fact_raw.shape)



Fact_Ventas RAW (con defectos): (60900, 9)


### 8.2 Dim_Producto — defectos
- **Texto inconsistente en `categoria`**: mayúsculas/minúsculas variables y espacios sobrantes, con énfasis en las categorías "Accesorios", "Laptops" y "Celulares" (variantes tipo "Accesorios"/"accesorios"/"ACCESORIOS").
- **Nulos en `categoria`** (~6%).

In [22]:
# ---------- Dim_Producto RAW ----------
dim_producto_raw = dim_producto.drop(columns=["_peso_popularidad"]).copy()

categorias_con_typos = {"Accesorios", "Laptops", "Celulares"}

def texto_inconsistente(cat, rng):
    r = rng.random()
    espacios_extra = "  " if rng.random() < 0.15 else ""
    if cat in categorias_con_typos:
        if r < 0.34:
            return espacios_extra + cat.upper() + espacios_extra
        elif r < 0.67:
            return espacios_extra + cat.lower() + espacios_extra
        else:
            return espacios_extra + cat + espacios_extra
    else:
        # inconsistencia leve de mayus/minus y espacios en las demas categorias
        if r < 0.15:
            return espacios_extra + cat.upper() + espacios_extra
        elif r < 0.30:
            return espacios_extra + cat.lower() + espacios_extra
        else:
            return espacios_extra + cat + espacios_extra

rng_txt = np.random.default_rng(SEED + 1)
dim_producto_raw["categoria"] = dim_producto_raw["categoria"].apply(lambda c: texto_inconsistente(c, rng_txt))

# valores faltantes ~6% en categoria
n_null_cat = int(len(dim_producto_raw) * 0.06)
idx_null_cat = np.random.choice(dim_producto_raw.index, size=n_null_cat, replace=False)
dim_producto_raw.loc[idx_null_cat, "categoria"] = np.nan

print("Dim_Producto RAW (con defectos):", dim_producto_raw.shape)


Dim_Producto RAW (con defectos): (500, 7)


### 8.3 Dim_Cliente — defectos
- **Nulos en `distrito`** (~7%).
- **Fechas en formatos mixtos** en `fecha_nacimiento` y `fecha_alta`.

Nota: la versión `raw` de `Dim_Cliente` **no** incluye la columna analítica `abandono` (esa columna es derivada y solo vive en la versión `processed`).

In [23]:
# ---------- Dim_Cliente RAW ----------
dim_cliente_raw = dim_cliente.copy()  # version raw NO lleva la columna 'abandono' (es derivada/analitica)

# valores faltantes ~7% en distrito
n_null_dist = int(len(dim_cliente_raw) * 0.07)
idx_null_dist = np.random.choice(dim_cliente_raw.index, size=n_null_dist, replace=False)
dim_cliente_raw.loc[idx_null_dist, "distrito"] = np.nan

# fechas en formatos mixtos
dim_cliente_raw["fecha_nacimiento"] = dim_cliente_raw["fecha_nacimiento"].apply(lambda f: formato_fecha_mixto(f, rng_dates))
dim_cliente_raw["fecha_alta"] = dim_cliente_raw["fecha_alta"].apply(lambda f: formato_fecha_mixto(f, rng_dates))

print("Dim_Cliente RAW (con defectos):", dim_cliente_raw.shape)


Dim_Cliente RAW (con defectos): (5000, 7)


### 8.4 Dim_Promocion, Dim_Tienda y Dim_Tiempo
`Dim_Promocion` recibe fechas en formatos mixtos (igual que las demás tablas con fechas). `Dim_Tienda` y `Dim_Tiempo` no tienen defectos específicos indicados en el requerimiento, por lo que su versión `raw` es igual a la limpia (se guardan en ambas carpetas para mantener las 6 tablas completas en `data/raw/`).

In [24]:
# ---------- Dim_Promocion RAW ----------
dim_promocion_raw = dim_promocion.copy()
dim_promocion_raw["fecha_inicio"] = dim_promocion_raw["fecha_inicio"].apply(lambda f: formato_fecha_mixto(f, rng_dates))
dim_promocion_raw["fecha_fin"] = dim_promocion_raw["fecha_fin"].apply(lambda f: formato_fecha_mixto(f, rng_dates))

# ---------- Dim_Tienda / Dim_Tiempo RAW ----------
# no se les inyectan defectos especificos en el enunciado; se guardan igual a la version limpia
dim_tienda_raw = dim_tienda.copy()
dim_tiempo_raw = dim_tiempo.drop(columns=["_peso_estacional"]).copy()

print("\nListo: todas las tablas RAW generadas.")



Listo: todas las tablas RAW generadas.


## 9. Exportación a CSV
Guardamos las 6 tablas **con problemas de calidad** en `data/raw/`, y las tablas de dimensión **limpias** (incluyendo `Dim_Cliente` con la columna `abandono`) en `data/processed/`. Todos los CSV usan `;` como separador y codificación UTF-8, tal como lo pide el requerimiento.

In [26]:
# =================================================================
# EXPORTACION A CSV
# =================================================================

def guardar_csv(df, ruta):
    df.to_csv(ruta, sep=";", index=False, encoding="utf-8")

# --- RAW (con problemas de calidad) ---
guardar_csv(dim_cliente_raw, f"{RAW_DIR}/dim_cliente.csv")
guardar_csv(dim_producto_raw, f"{RAW_DIR}/dim_producto.csv")
guardar_csv(dim_tienda_raw, f"{RAW_DIR}/dim_tienda.csv")
guardar_csv(dim_tiempo_raw, f"{RAW_DIR}/dim_tiempo.csv")
guardar_csv(dim_promocion_raw, f"{RAW_DIR}/dim_promocion.csv")
guardar_csv(fact_raw, f"{RAW_DIR}/fact_ventas.csv")

# --- PROCESSED (Dim_* limpias) ---
#guardar_csv(dim_cliente_clean, f"{PROC_DIR}/dim_cliente.csv")
#guardar_csv(dim_producto.drop(columns=["_peso_popularidad"]), f"{PROC_DIR}/dim_producto.csv")
#guardar_csv(dim_tienda, f"{PROC_DIR}/dim_tienda.csv")
#guardar_csv(dim_tiempo.drop(columns=["_peso_estacional"]), f"{PROC_DIR}/dim_tiempo.csv")
#guardar_csv(dim_promocion, f"{PROC_DIR}/dim_promocion.csv")

print("\nArchivos CSV guardados en data/raw/ y data/processed/.")



Archivos CSV guardados en data/raw/ y data/processed/.


## 10. Resumen de calidad inicial
Por último, generamos un resumen que evidencia el estado inicial de calidad de los datos crudos: número de filas por tabla, porcentaje de nulos por columna, cantidad de líneas duplicadas y cantidad de claves huérfanas en `Fact_Ventas`. Este resumen es el punto de partida típico de un proceso de limpieza / ETL.

In [28]:
# =================================================================
# RESUMEN DE CALIDAD INICIAL
# =================================================================

tablas_raw = {
    "Dim_Cliente": dim_cliente_raw,
    "Dim_Producto": dim_producto_raw,
    "Dim_Tienda": dim_tienda_raw,
    "Dim_Tiempo": dim_tiempo_raw,
    "Dim_Promocion": dim_promocion_raw,
    "Fact_Ventas": fact_raw,
}

print("\n" + "=" * 70)
print("RESUMEN DE CALIDAD INICIAL - data/raw/")
print("=" * 70)

for nombre, df in tablas_raw.items():
    print(f"\n--- {nombre} ---")
    print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")
    pct_nulos = (df.isna().mean() * 100).round(2)
    pct_nulos = pct_nulos[pct_nulos > 0]
    if len(pct_nulos) > 0:
        print("Porcentaje de nulos por columna:")
        for col, pct in pct_nulos.items():
            print(f"   - {col}: {pct}%")
    else:
        print("Sin valores nulos.")

# duplicados en Fact_Ventas: lineas cuyo id_venta aparece mas de una vez
# (nota: algunas mutaciones posteriores -como nulos en descuento u outliers-
# pueden aplicarse de forma independiente a cada copia, por lo que ademas
# de duplicados exactos pueden existir "casi duplicados")
n_duplicados = fact_raw["id_venta"].duplicated().sum()
print(f"\nDuplicados detectados en Fact_Ventas (id_venta repetido): {n_duplicados:,}")

# claves huerfanas: id_producto en Fact_Ventas que no existe en Dim_Producto
ids_validos = set(dim_producto["id_producto"])
n_huerfanas_final = (~fact_raw["id_producto"].isin(ids_validos)).sum()
print(f"Claves huerfanas detectadas (id_producto inexistente en Dim_Producto): {n_huerfanas_final:,}")

print("\n" + "=" * 70)
print("Generacion de datos sinteticos completada.")
print("=" * 70)



RESUMEN DE CALIDAD INICIAL - data/raw/

--- Dim_Cliente ---
Filas: 5,000  |  Columnas: 7
Porcentaje de nulos por columna:
   - distrito: 7.0%

--- Dim_Producto ---
Filas: 500  |  Columnas: 7
Porcentaje de nulos por columna:
   - categoria: 6.0%

--- Dim_Tienda ---
Filas: 15  |  Columnas: 5
Sin valores nulos.

--- Dim_Tiempo ---
Filas: 730  |  Columnas: 7
Sin valores nulos.

--- Dim_Promocion ---
Filas: 40  |  Columnas: 6
Sin valores nulos.

--- Fact_Ventas ---
Filas: 60,900  |  Columnas: 9
Porcentaje de nulos por columna:
   - id_promocion: 54.0%
   - descuento: 5.0%

Duplicados detectados en Fact_Ventas (id_venta repetido): 900
Claves huerfanas detectadas (id_producto inexistente en Dim_Producto): 307

Generacion de datos sinteticos completada.
